**Necessary imports**

In [1]:
import subprocess
import os
import pandas as pd
import json

%load_ext autoreload
%autoreload 2

**Setting parameters**

In [2]:
simple_render_raw_data_dir = "/dlabdata1/tsoares/wikidumps/"
root_processing_dir = "/dlabdata1/boulenge/wikidumps_data"
experiment_name = "selected_langs_small_sample"
processed_data_dir = os.path.join(root_processing_dir, experiment_name)
server_out_dir = os.path.join("/var/www/html/wikinsert", experiment_name)
processes = 48
min_nb_samples = 10
revisions_sampling_size = 0.0005 #10% of the revisions 

#For all possible languages
all_langs = sorted(set([filename.split("wiki")[0] for filename in os.listdir(simple_render_raw_data_dir) if "wiki" in filename]))

selected_langs_path = 'selected_languages.json'

with open(selected_langs_path, 'r') as f:
   langs = json.load(f)

side_context_sizes = [5]  #List of context size to try
second_month = "20231001"
third_month = "20231101"

langs_to_remove = ["war", "ky", "nds-nl"] #These languages do not work

for lang in langs_to_remove:
    if lang in all_langs:
        all_langs.remove(lang)

print("Will process languages:\n" + '\n'.join(langs))

Will process languages:
af
ar
cs
cy
de
en
es
fa
fr
ga
gu
he
hi
is
it
ja
jv
kk
kn
ms
pt
ru
simple
sk
sw
tr
ur
uz
vi
zh


**Defining data paths**

In [6]:
# Define directories
all_revisions_dir = os.path.join(root_processing_dir, "all_revisions")
rev_files_dir = os.path.join(processed_data_dir, "revisions_files")
parsed_htmls_dir = os.path.join(processed_data_dir, "parsed_htmls")
contexts_splitted_data_dir = os.path.join(processed_data_dir, "contexts_splitted_data")
ready_to_score_data_dir = os.path.join(processed_data_dir, "ready_to_score_data")
scored_data_dir = os.path.join(processed_data_dir, "scored_data")
crawling_output_dir = os.path.join(processed_data_dir, "crawled_pages")

# Create necessary directories if they don't exist
os.makedirs(all_revisions_dir, exist_ok=True)
os.makedirs(rev_files_dir, exist_ok=True)
os.makedirs(parsed_htmls_dir, exist_ok=True)
os.makedirs(contexts_splitted_data_dir, exist_ok=True)
os.makedirs(ready_to_score_data_dir, exist_ok=True)
os.makedirs(scored_data_dir, exist_ok=True)
os.makedirs(crawling_output_dir, exist_ok=True)
os.makedirs(server_out_dir, exist_ok=True)

**Prepare files to be crawled and parsed**

**Carefull, the following is only needed to be ran once for all (common for any experiment), so was probably ran before**

It generates the diffenret pages ids we will crawl, using data from the previous study

In [22]:
"""for lang in all_langs:
        print(f"Processing language: {lang}")
        second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
        third_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{third_month}"
        save_path = f"{all_revisions_dir}/revisions_{lang}.parquet"
        if not os.path.exists(save_path):
            print("Generating revisions file")    
            subprocess.run([
                "python", "generate_revisions_file.py",
                "--versions_file", f"{second_month_path}/eval/link_versions",
                "--html_dir", f"{second_month_path}/eval/pages",
                "--redirect_1", f"{second_month_path}/processed_data/redirect_map.parquet",
                "--redirect_2", f"{third_month_path}/processed_data/redirect_map.parquet",
                "--mention_map", f"{second_month_path}/processed_data/mention_map.parquet",
                "--save_path", save_path
            ])
        else:
            print("Revisions file already exists")"""


'for lang in all_langs:\n        print(f"Processing language: {lang}")\n        second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"\n        third_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{third_month}"\n        save_path = f"{all_revisions_dir}/revisions_{lang}.parquet"\n        if not os.path.exists(save_path):\n            print("Generating revisions file")    \n            subprocess.run([\n                "python", "generate_revisions_file.py",\n                "--versions_file", f"{second_month_path}/eval/link_versions",\n                "--html_dir", f"{second_month_path}/eval/pages",\n                "--redirect_1", f"{second_month_path}/processed_data/redirect_map.parquet",\n                "--redirect_2", f"{third_month_path}/processed_data/redirect_map.parquet",\n                "--mention_map", f"{second_month_path}/processed_data/mention_map.parquet",\n                "--save_path", save_path\n            ])\n        else:\n 

**Sampling**

In [23]:
langs_stats = []

for lang in langs:
    full_revisions_path = f"{all_revisions_dir}/revisions_{lang}.parquet"
    full_revisions_df = pd.read_parquet(full_revisions_path)
    print(f"Sampling language: {lang}")
    nb_total = len(full_revisions_df)
    pre_min_to_sample = 2*min_nb_samples #we take twice the number to prevent future loss of data
    if nb_total > pre_min_to_sample:
        pre_nb_samples = max(pre_min_to_sample, int(revisions_sampling_size*2*nb_total))
    else:
        pre_nb_samples = nb_total

    if nb_total > min_nb_samples:
        nb_samples = max(min_nb_samples, int(revisions_sampling_size*nb_total))
    else:
        nb_samples = nb_total

    langs_stats.append({
        "lang": lang,
        "orig_nb_revisions": nb_total,
        "pre_nb_samples": pre_nb_samples,
        "nb_samples": nb_samples
    })

    sampled_revisions_path = f"{rev_files_dir}/revisions_{lang}.parquet"
    sampled_revisions = full_revisions_df.sample(n=pre_nb_samples, random_state=42)
    sampled_revisions.to_parquet(sampled_revisions_path)

display(pd.DataFrame(langs_stats))
display(pd.DataFrame(langs_stats).sum())


Sampling language: af
Sampling language: ar
Sampling language: cs
Sampling language: cy
Sampling language: de
Sampling language: en
Sampling language: es
Sampling language: fa
Sampling language: fr
Sampling language: ga
Sampling language: gu
Sampling language: he
Sampling language: hi
Sampling language: is
Sampling language: it
Sampling language: ja
Sampling language: jv
Sampling language: kk
Sampling language: kn
Sampling language: ms
Sampling language: pt
Sampling language: ru
Sampling language: simple
Sampling language: sk
Sampling language: sw
Sampling language: tr
Sampling language: ur
Sampling language: uz
Sampling language: vi
Sampling language: zh


,lang,orig_nb_revisions,pre_nb_samples,nb_samples
0,af,707,20,10
1,ar,6700,20,10
2,cs,4391,20,10
3,cy,238,20,10
4,de,49899,49,24
5,en,194300,194,97
6,es,29970,29,14
7,fa,7967,20,10
8,fr,26847,26,13
9,ga,109,20,10


lang                 afarcscydeenesfafrgaguhehiisitjajvkkknmsptrusi...
orig_nb_revisions                                               467883
pre_nb_samples                                                     832
nb_samples                                                         417
dtype: object

**Crawling first versions only**

There is no need to re-crawl the second version with all its files as the index is sufficent for these.

In [7]:
for lang in langs:
    print(f"Processing language: {lang}")
    subprocess.run([
        "python", "crawl.py",
        "--revisions_path", f"{rev_files_dir}/revisions_{lang}.parquet",
        "--output_dir", f"{crawling_output_dir}/{lang}",
        "--n_processes", str(processes),
        "--lang", lang
    ])

Processing language: af
Total number of pages to crawl: 20
Running crawling on 48 processes
Crawling finished in 0.15749736626942953 minutes, post processing...


Moving files: 100%|██████████| 20/20 [00:00<00:00, 135.38it/s]


Processing language: ar
Total number of pages to crawl: 20
Running crawling on 48 processes
Crawling finished in 0.3681405504544576 minutes, post processing...


Moving files: 100%|██████████| 20/20 [00:00<00:00, 75.03it/s]


Processing language: cs
Total number of pages to crawl: 20
Running crawling on 48 processes
Crawling finished in 0.12054530779520671 minutes, post processing...


...

Moving files: 100%|██████████| 20/20 [00:00<00:00, 196.03it/s]


Total number of pages to crawl: 20
Running crawling on 48 processes
Crawling finished in 0.13142793575922648 minutes, post processing...


Moving files: 100%|██████████| 20/20 [00:00<00:00, 161.20it/s]


Processing language: zh
Total number of pages to crawl: 20
Running crawling on 48 processes
Crawling finished in 0.15469912687937418 minutes, post processing...


Moving files: 100%|██████████| 20/20 [00:00<00:00, 160.31it/s]


**Parsing of the htmls and sentence tokenizing**

In [9]:
for lang in langs:
    print(f"Processing language: {lang}")
    second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
    first_version_specific_html_dir = f"{crawling_output_dir}/{lang}"
    save_path = f"{rev_files_dir}/revisions_{lang}.parquet"
    output_dir = f"{parsed_htmls_dir}/{lang}_v2"
    first_version_specific_output_dir = f"{parsed_htmls_dir}/{lang}_v1"
        
    subprocess.run([
        "python", "parsing.py",
        "--revisions_path", save_path,        
        "--html_dir", f"{second_month_path}/eval/pages",
        "--first_versions_specific_html_dir", first_version_specific_html_dir,
        "--output_dir", output_dir,
        "--first_versions_specific_output_dir", first_version_specific_output_dir,
        "--lang", lang,
        "--n_processes", str(processes)
    ])

    parsed_htmls_paths = os.listdir(first_version_specific_output_dir)
    sentences_path = list(
        filter(
            lambda x: (
                x.endswith("sentences.jsonl") and ("raw" not in x)
                ),
            parsed_htmls_paths
            )
        )[0]
    links_path = list(filter(lambda x: x.endswith("links.jsonl"), parsed_htmls_paths))[0]
    sentences_df = pd.read_json(f"{first_version_specific_output_dir}/{sentences_path}")
    links_df = pd.read_json(f"{first_version_specific_output_dir}/{links_path}")

    display(sentences_df.head(5))
    display(links_df.head(5)) 

Processing language: af
First version html dir: /var/www/html/wikinsert/selected_langs_small_sample/crawled_pages/af
Running on 48 processes
Parallel multiprocess
Writing sentences and links v1


Sentence tokenization: 100%|██████████| 20/20 [00:00<00:00, 59.78it/s]


Time taken for v1: 15.523229360580444
Writing sentences and links v2


Sentence tokenization: 100%|██████████| 20/20 [00:00<00:00, 61.54it/s]


,md5,id,text,startIdx,endIdx,section,locList
0,1509afbe912fc0e70ec3d7a083f7e21b,0,Piket-Bo-Berg is 'n plaasgemeenskap geleë in d...,34755,34882,Lead,"[{'start': 34755, 'end': 34768}, {'start': 347..."
1,c771bb69375660fb63a2c352317aafc7,1,Die gemeenskap lê bo-op die Piketberge en is n...,34882,35033,Lead,"[{'start': 34882, 'end': 34950}, {'start': 349..."
2,47b40cd8340f99c7be3ce1ea8b867e63,2,Dit is oor tyd deur 'n verskeidenheid boere be...,35033,35145,Lead,"[{'start': 35033, 'end': 35145}]"
3,49a811f19c48f823bf7f53ecdee39c16,3,"Voorheen is die Bo-Berg bewoon deur Boesmanne,...",35145,35239,Lead,"[{'start': 35145, 'end': 35239}]"
4,166bf21d3b5aa648f209a64e799810a6,4,In 1810 was daar vyf plase wat daar op die Ber...,35239,35307,Lead,"[{'start': 35246, 'end': 35307}]"


...

Sentence tokenization: 100%|██████████| 20/20 [00:00<00:00, 55.18it/s]


Time taken for v1: 31.38294553756714
Writing sentences and links v2


Sentence tokenization: 100%|██████████| 20/20 [00:00<00:00, 56.32it/s]


,md5,id,text,startIdx,endIdx,section,locList
0,a87b7f9939b52e11c45b15832a61fbe3,0,"\n未入流是明、清兩代對於低於從九品的官吏的稱呼，在唐代則被稱為""流外""。",42371,42477,Lead,"[{'start': 42370, 'end': 42371}, {'start': 423..."
1,76db8e7fe8a7329105d1cc465748b043,1,\n\n\n清代:\n文官\n\n\n\n武官\n\n\n\n《明史．卷七二．職官志一》：「...,42477,54907,文獻提及,"[{'start': 42481, 'end': 42482}, {'start': 425..."
2,decd6e782d8676fe47b96b8b6eb645cf,2,不及九品曰未入流。,54907,54916,文獻提及,"[{'start': 54907, 'end': 54916}]"
3,15923bd3fe492768a389e727b2a3dd82,3,」。,54916,54918,文獻提及,"[{'start': 54916, 'end': 54918}]"
4,f0948063ed2bfddf09570c7c8df17e4c,4,\n《二十年目睹之怪現狀》第二四回：“我屈著指頭算道：「降級是降正不降從的，降一級便是八品，...,54918,55064,文獻提及,"[{'start': 54993, 'end': 54994}, {'start': 550..."


,link,startIdx,text,endIdx,section,sentid
0,file:///wiki/%E7%83%8F%E5%85%8B%E8%98%AD%E8%AA%9E,44844,烏克蘭語,44938,Lead,0
1,file:///wiki/%E7%83%8F%E5%85%8B%E8%98%AD,44972,烏克蘭,45055,Lead,0
2,file:///wiki/%E5%88%87%E7%88%BE%E5%B0%BC%E6%88...,45065,切爾尼戈夫州,45161,Lead,0
3,file:///wiki/%E4%BC%8A%E5%A5%87%E5%B0%BC%E4%BA...,45163,伊奇尼亞區,45248,Lead,0
4,file:///wiki/%E4%BA%92%E8%81%94%E7%BD%91%E6%A1...,45746,互联网档案馆,45842,參考資料,1


**Preprocessing**

It included all prperocessing parts but the merging

In [12]:
for lang in langs:
    for side_context_size in side_context_sizes:
        print(f"Processing language: {lang} with side context size {side_context_size}")
        second_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{second_month}"
        third_month_path = f"{simple_render_raw_data_dir}/{lang}wiki-NS0-{third_month}"
        save_path = f"{rev_files_dir}/revisions_{lang}.parquet"
        output_dir = f"{contexts_splitted_data_dir}/{lang}/context_size_{side_context_size}"
        
        subprocess.run([
            "python", "preprocessing.py",
            "--revisions_path", save_path,
            "--parsed_html_dir", f"{parsed_htmls_dir}/{lang}_v2",
            "--first_versions_specific_parsed_html_dir", f"{parsed_htmls_dir}/{lang}_v1",
            "--n_processes", str(processes),
            "--output_dir", output_dir,
            "--side_contexts_size", str(side_context_size),
            "--redirect_1", f"{second_month_path}/processed_data/redirect_map.parquet",
            "--redirect_2", f"{third_month_path}/processed_data/redirect_map.parquet"
        ])

        #preprocessed_data_path = os.path.join(output_dir, f'test_data.parquet')

        #preprocessed_data = pd.read_parquet(preprocessed_data_path)
        #display(preprocessed_data.head(5))

Processing language: af with side context size 5
Running on 48 processes


100%|██████████| 83/83 [00:00<00:00, 3274.92it/s]


The length of the output is 83
Processing language: ar with side context size 5
Running on 48 processes


100%|██████████| 20/20 [00:07<00:00,  2.56it/s]


The length of the output is 38


...

100%|██████████| 66/66 [00:00<00:00, 7487.40it/s]


Processing language: zh with side context size 5
Running on 48 processes


100%|██████████| 20/20 [00:09<00:00,  2.17it/s]


The length of the output is 89


100%|██████████| 89/89 [00:00<00:00, 55033.62it/s]


**Merging**

In [13]:
test_data_dir = contexts_splitted_data_dir
output_dir = ready_to_score_data_dir

subprocess.run([
    "python", "merge_data.py",
    "--test_data_dir", test_data_dir,
    "--raw_data_dir", simple_render_raw_data_dir,
    "--langs", ",".join(langs),
    "--side_contexts_sizes", ",".join(map(str, side_context_sizes)),
    "--output_dir", output_dir,
    "--ignore_warnings"
])

first_lang = langs[0]
first_context_size = str(side_context_sizes[0])
scored_files_example = os.listdir(output_dir)
scored_files_example = list(filter(lambda x: (first_lang in x and first_context_size in x), scored_files_example))
final_input_data_path = os.path.join(output_dir, scored_files_example[0])

final_input_data_df = pd.read_parquet(final_input_data_path)
display(final_input_data_df.head(5))

Saving mention maps:
Processing language af (1/30)
Finished processing language af
Processing language ar (2/30)
Finished processing language ar
Processing language cs (3/30)
Finished processing language cs
Processing language cy (4/30)
Finished processing language cy
Processing language de (5/30)
Finished processing language de
Processing language en (6/30)
Finished processing language en
Processing language es (7/30)
Finished processing language es
Processing language fa (8/30)
Finished processing language fa
Processing language fr (9/30)
Finished processing language fr
Processing language ga (10/30)
Finished processing language ga
Processing language gu (11/30)
Finished processing language gu
Processing language he (12/30)
Finished processing language he
Processing language hi (13/30)
Finished processing language hi
Processing language is (14/30)
Finished processing language is
Processing language it (15/30)
Finished processing language it
Processing language ja (16/30)
Finished pro

100%|██████████| 110890/110890 [00:00<00:00, 709658.10it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 83 links for language af
Removed 0 links because of empty contexts
Removed 23 links because target title is not in source titles
Removed 21 links because of missing section
After cleaning, there are 39 links for language af
Finished processing language af
Processing language ar (2/30)
Loaded pages
Loaded redirects


100%|██████████| 1209639/1209639 [00:01<00:00, 811869.17it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 38 links for language ar
Removed 0 links because of empty contexts
Removed 6 links because target title is not in source titles
Removed 7 links because of missing section
After cleaning, there are 25 links for language ar
Finished processing language ar
Processing language cs (3/30)
Loaded pages
Loaded redirects


...

100%|██████████| 1273459/1273459 [00:01<00:00, 879209.50it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 66 links for language vi
Removed 0 links because of empty contexts
Removed 11 links because target title is not in source titles
Removed 23 links because of missing section
After cleaning, there are 32 links for language vi
Finished processing language vi
Processing language zh (30/30)
Loaded pages
Loaded redirects


100%|██████████| 1344667/1344667 [00:01<00:00, 800066.24it/s]


Processing context size 5
Loaded links for context_size_5
Originally, there are 89 links for language zh
Removed 0 links because of empty contexts
Removed 11 links because target title is not in source titles
Removed 3 links because of missing section
After cleaning, there are 75 links for language zh
Finished processing language zh


,source_title,source_ID,target_title,context,insert_indices,moved_insertion_marker,link_sentence,link_text,section,mention_present,source_lead,first_version,second_version,direct_match,missing_category,all_contexts,target_lead
0,1942,486,9_Julie,None,[62],None,"9 Julie - Richard Roundtree, Amerikaanse akteu...",9 Julie,Geboortes,None,Die jaar 1942 was 'n gewone jaar wat volgens d...,2622176,2624839,None,missing_sentence,[{'context': 'Die jaar 1942 was 'n gewone jaar...,9 Julie is die 190ste dag van die jaar in die ...
1,Marius_Petipa,407068,Pjotr_Tsjaikofski,None,[167],None,Gedurende die goue era van die Russiese Ryk he...,Tsjaikofski,"Sint Petersburg, Rusland",None,Marius Ivanovich Petipa (Russies: Мариус Ивано...,2621414,2621682,None,missing_span,[{'context': 'Marius Ivanovich Petipa (Russies...,"Pjotr Iljitsj Tsjaikofski, Wotkinsk, 7 Mei 184..."
2,Marius_Petipa,407068,Groothertog_Aleksander_Michailowitsj_van_Rusland,None,[167],None,Gedurende die goue era van die Russiese Ryk he...,Groothertog Aleksander Michailowitsj van Rusland,"Sint Petersburg, Rusland",None,Marius Ivanovich Petipa (Russies: Мариус Ивано...,2621414,2621682,None,missing_span,[{'context': 'Marius Ivanovich Petipa (Russies...,Hierdie naam is volgens die Oos-Slawiese gebru...
3,Marius_Petipa,407068,Nikolaas_II_van_Rusland,None,[167],None,Gedurende die goue era van die Russiese Ryk he...,Keiser Nikolaas II,"Sint Petersburg, Rusland",None,Marius Ivanovich Petipa (Russies: Мариус Ивано...,2621414,2621682,None,missing_span,[{'context': 'Marius Ivanovich Petipa (Russies...,Nikolaas II Aleksandrowitsj was die laaste kei...
4,Marius_Petipa,407068,Giuseppe_Verdi,None,[167],None,Gedurende die goue era van die Russiese Ryk he...,Verdi,"Sint Petersburg, Rusland",None,Marius Ivanovich Petipa (Russies: Мариус Ивано...,2621414,2621682,None,missing_span,[{'context': 'Marius Ivanovich Petipa (Russies...,Giuseppe Fortunino Francesco Verdi was 'n Ital...


**Scoring the contexts**

In [4]:
data_dir = ready_to_score_data_dir
output_dir = scored_data_dir

for lang in langs:
    for side_context_size in side_context_sizes:
        print(f"Processing language: {lang} with side context size {side_context_size}")

        subprocess.run([
            "python", "get_llm_scores.py",
            "--data_dir", data_dir,
            "--output_dir", output_dir,
            "--lang", lang,
            "--side_contexts_size", str(side_context_size),
            "--model_dir", "/dlabdata1/tsoares/models",
            "--model_prefix", "roberta_full",
            "--use_section_title",
            "--batch_size", "32",
            "--use_mentions",
            "--use_cuda",
            "--multilingual",
            "--multilingual_name", "multilingual-all"
        ])

scored_files_example = os.listdir(output_dir)[0]
scored_data = pd.read_parquet(f"{output_dir}/{scored_files_example}")

exemple = scored_data['all_contexts'][0][0]

for key in exemple.keys():
    print(f"{key}:\n{exemple[key]}\n")

Processing language: simple with side context size 5


Traceback (most recent call last):
  File "/scratch/boulenge/wikinsert/get_llm_scores.py", line 112, in <module>
    raise ValueError(f'{lang} already done for context_size {side_contexts_size} and model {full_model_name}, passing')
ValueError: simple already done for context_size 5 and model roberta_full_multilingual-all, passing


Processing language: sk with side context size 5


Traceback (most recent call last):
  File "/scratch/boulenge/wikinsert/get_llm_scores.py", line 112, in <module>
    raise ValueError(f'{lang} already done for context_size {side_contexts_size} and model {full_model_name}, passing')
ValueError: sk already done for context_size 5 and model roberta_full_multilingual-all, passing


Processing language: sw with side context size 5


...

Processing language: vi with side context size 5


Traceback (most recent call last):
  File "/scratch/boulenge/wikinsert/get_llm_scores.py", line 112, in <module>
    raise ValueError(f'{lang} already done for context_size {side_contexts_size} and model {full_model_name}, passing')
ValueError: vi already done for context_size 5 and model roberta_full_multilingual-all, passing


Processing language: zh with side context size 5


Traceback (most recent call last):
  File "/scratch/boulenge/wikinsert/get_llm_scores.py", line 112, in <module>
    raise ValueError(f'{lang} already done for context_size {side_contexts_size} and model {full_model_name}, passing')
ValueError: zh already done for context_size 5 and model roberta_full_multilingual-all, passing


context:
Vòng bảng UEFA Europa Conference League 2023–24 bắt đầu vào ngày 21 tháng 9 năm 2023 và kết thúc vào ngày 14 tháng 12 năm 2023. Có tổng cộng 32 đội thi đấu ở vòng bảng để xác định 16 trong số 24 suất vào vòng đấu loại trực tiếp của UEFA Europa Conference League 2023–24. Tất cả các đội bóng ngoài AZ, Ballkani, Bodø/Glimt, Fiorentina, Gent, HJK, Maccabi Tel Aviv, PAOK, Slovan Bratislava và Zorya Luhansk có lần đầu tiên ra mắt ở vòng bảng. Breiðablik, Čukarički, KÍ, Olimpija Ljubljana và Zrinjski Mostar có lần đầu tiên ra mắt ở vòng bảng một giải đấu UEFA. Breiðablik, KÍ và Zrinjski Mostar là những đội bóng đầu tiên lần lượt từ Iceland, Quần đảo Faroe và Bosnia và Herzegovina thi đấu ở vòng bảng một giải đấu UEFA. Có tổng cộng 28 hiệp hội quốc gia được đại diện ở vòng bảng.

middle_sentence:
{'clean_sentence': 'Vòng bảng UEFA Europa Conference League 2023–24 bắt đầu vào ngày 21 tháng 9 năm 2023 và kết thúc vào ngày 14 tháng 12 năm 2023.', 'clean_sentence2': None, 'endIdx1': 49121

**Colouring the pages**

In [24]:
sorted_link_output_dir = os.path.join(server_out_dir, "colored_pages")
soft_links_dir = os.path.join(server_out_dir, "soft_links")
print()
# Run the coloring script
subprocess.run([
    "python", "color_pages.py",
    "--scored_data_dir", scored_data_dir,
    "--html_dir", crawling_output_dir,
    "--sorted_links_output_dir", sorted_link_output_dir,
    "--soft_links_dir", soft_links_dir,
    "--langs", ",".join(langs),
    "--side_contexts_sizes", ",".join(map(str, side_context_sizes)),
    "--extra_crawled_files_dir", crawling_output_dir,
    "--full_model_name", "roberta_full_multilingual-all"
])
